In [1]:
# import modules
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import h5py as h5

import holopy as hp
from holopy.core.process import normalize, bg_correct, center_find, subimage
from holopy.scattering import Sphere, Spheres, calc_holo
from holopy.inference import prior, ExactModel, CmaStrategy, EmceeStrategy, AlphaModel, NmpfitStrategy, KaiModel

[dhcp-10-250-168-9.harvard.edu:97623] shmem: mmap: an error occurred while determining whether or not /var/folders/b9/qjpfvnt53313gkrkg4lhksmr0000gn/T//ompi.dhcp-10-250-168-9.501/jf.0/691929088/sm_segment.dhcp-10-250-168-9.501.293e0000.0 could be created.


ImportError: cannot import name 'KaiModel' from 'holopy.inference' (/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holopy_code/holopy/holopy/inference/__init__.py)

In [25]:
#needed to make display work properly (there are other options as well if this fails)
%matplotlib tk

In [26]:
# ground truth parameters used to generate a hologram using scattering theory
# optical parameters
medium_index = 1.33
illum_wavelen = 0.660
illum_polarization = (0.56, 0.83)
detector = hp.detector_grid(shape=100, spacing=0.177)

# geometric parameters
N_1_TRUE = 1.59
N_2_TRUE = 1.59
R_1_TRUE = 0.65
R_2_TRUE = 0.65
X1_TRUE = 5
Y1_TRUE = 5
Z1_TRUE = 5
X2_TRUE = 5
Y2_TRUE = 5
Z2_TRUE = 3.5
Xg_TRUE = (X1_TRUE + X2_TRUE)/2
Yg_TRUE = (Y1_TRUE + Y2_TRUE)/2
Zg_TRUE = (Z1_TRUE + Z2_TRUE)/2

SAVEPATH = '/Users/kaitorrens/harvard_grad_school/manoharan_lab/Code/holographic-potential-measurement/kai_new_fitting/kai_fits/von_Mises_Fisher_testing_1_'

In [27]:
# create a hologram from two spheres, one above the other
s1 = Sphere(center=(X1_TRUE, Y1_TRUE, Z1_TRUE), n = N_1_TRUE, r = R_1_TRUE)
s2 = Sphere(center=(X2_TRUE, Y2_TRUE, Z2_TRUE), n = N_2_TRUE, r = R_2_TRUE)

collection = Spheres([s1, s2])
holo1 = calc_holo(detector, collection, medium_index, illum_wavelen,
                 illum_polarization)
hp.show(holo1)

In [11]:
# create a hologram from two spheres, one next to the other
s1 = Sphere(center=(5, 5, 5), n = N_1_TRUE, r = R_1_TRUE)
s2 = Sphere(center=(4, 4, 5), n = N_2_TRUE, r = R_2_TRUE)

collection = Spheres([s1, s2])
holo2 = calc_holo(detector, collection, medium_index, illum_wavelen,
                 illum_polarization)
hp.show(holo2)

In [28]:
# info for modelling/ fitting

SPACING = 0.177
WAVELEN = 0.660
MEDIUM_INDEX = 1.33
POLARIZATION = POLARIZATION = [0.56, 0.83] # Calibrated 2020-09-02

#Sphere 1
R_1_MEAN = 0.6749016953839639
R_1_SIGMA =  0.00047233977835876834
N_1_MEAN =  1.5848484802283918
N_1_SIGMA =  0.00027320846407879903
#Sphere 2
R_2_MEAN =  0.6446649533295826
R_2_SIGMA =  0.000617682113114549
N_2_MEAN =  1.601784237444771
N_2_SIGMA =  0.0003353994780766957
DIMER_Z_GUESS = 4.20

In [20]:
def create_model(parameters):
    s1_r = parameters['r_1']
    s2_r = parameters['r_2']
    s1_n = parameters['n_1']
    s2_n = parameters['n_2']
    center_x = parameters['x_g']
    center_y = parameters['y_g']
    center_z = parameters['z_g']
    theta = parameters['theta']
    phi = parameters['phi']
    gap = parameters['gap']
    alpha = parameters['alpha']
    
    gap_center = np.array([center_x, center_y, center_z])
    components = np.array([np.cos(phi) * np.sin(theta), np.sin(phi) * np.sin(theta), np.cos(theta)])
    s1_center = gap_center + (s1_r + gap/2) * components
    s2_center = gap_center - (s2_r + gap/2) * components
        
    scatterer = Spheres([Sphere(r=s1_r, n=s1_n, center=s1_center),
                         Sphere(r=s2_r, n=s2_n, center=s2_center)], warn=False)
    return AlphaModel(scatterer, alpha=alpha)  


In [22]:
# try to fit hologram (adapted from Caroline's code)

dimer_holo = holo1
#x, y = img.center * SPACING
x, y = Xg_TRUE, Yg_TRUE

# Step 1: Fitting the dimer with gap set to 0
# Set priors and run CMAES allowing z, Theta, Phi and alpha to vary

# I may need to modify the bounds on these priors
z = prior.Uniform(2, 10, name="Z", guess=DIMER_Z_GUESS)
theta = prior.Uniform(0-0.1, np.pi+0.1, name="Theta")
phi = prior.Uniform(0-0.1, 2*np.pi + 0.1, name='Phi')
alpha = prior.Uniform(0.5, 1.2, name="Alpha", guess = 0.8)
step_2_parameters = {'r_1': R_1_MEAN, 'r_2': R_2_MEAN, 'n_1': N_1_MEAN, 'n_2': N_2_MEAN,
                    'x_g': x, 'y_g': y, 'z_g': z,
                    'theta': theta, 'phi': phi, 'gap': 0, 'alpha': alpha}
model =  create_model(step_2_parameters)
cma_fit_strategy = CmaStrategy(popsize=100, walker_initial_pos=model.generate_guess(scaling=0.5, n=100))
results2 = hp.fit(dimer_holo, model, strategy=cma_fit_strategy)
hp.save(SAVEPATH+'_nogap.h5', results2)

print('Independent angle fit completed')
print(results2.guess_parameters)
print(results2.parameters)

[dhcp-10-250-148-224.harvard.edu:60648] shmem: mmap: an error occurred while determining whether or not /var/folders/b9/qjpfvnt53313gkrkg4lhksmr0000gn/T//ompi.dhcp-10-250-148-224.501/jf.0/3817013248/sm_segment.dhcp-10-250-148-224.501.e3830000.0 could be created.
[dhcp-10-250-148-224.harvard.edu:60643] shmem: mmap: an error occurred while determining whether or not /var/folders/b9/qjpfvnt53313gkrkg4lhksmr0000gn/T//ompi.dhcp-10-250-148-224.501/jf.0/237699072/sm_segment.dhcp-10-250-148-224.501.e2b0000.0 could be created.
[dhcp-10-250-148-224.harvard.edu:60646] shmem: mmap: an error occurred while determining whether or not /var/folders/b9/qjpfvnt53313gkrkg4lhksmr0000gn/T//ompi.dhcp-10-250-148-224.501/jf.0/3601006592/sm_segment.dhcp-10-250-148-224.501.d6a30000.0 could be created.
[dhcp-10-250-148-224.harvard.edu:60642] shmem: mmap: an error occurred while determining whether or not /var/folders/b9/qjpfvnt53313gkrkg4lhksmr0000gn/T//ompi.dhcp-10-250-148-224.501/jf.0/146604032/sm_segment.dhcp

NameError: name 'SAVEPATH' is not defined

In [29]:
def create_model_angles(parameters):
    s1_r = parameters['r_1']
    s2_r = parameters['r_2']
    s1_n = parameters['n_1']
    s2_n = parameters['n_2']
    center_x = parameters['x_g']
    center_y = parameters['y_g']
    center_z = parameters['z_g']
    angles = parameters['angles']
    # problem with theta, phi when they are in a joint distribution
    # solution 1) somehow use just one object
    # angles = parameters['angles']
    # solution 2) somehow treat theta, phi as a vector object
    # theta, phi = parameters['angles']
    # solution 3) define priors that angles can be split up into
    #theta = parameters['angles'].theta
    #phi = parameters['angles'].phi
    gap = parameters['gap']
    alpha = parameters['alpha']
    theta, phi = angles
    
    gap_center = np.array([center_x, center_y, center_z])
    components = np.array([np.cos(phi) * np.sin(theta), np.sin(phi) * np.sin(theta), np.cos(theta)])
    s1_center = gap_center + (s1_r + gap/2) * components
    s2_center = gap_center - (s2_r + gap/2) * components
        
    scatterer = Spheres([Sphere(r=s1_r, n=s1_n, center=s1_center),
                         Sphere(r=s2_r, n=s2_n, center=s2_center)], warn=False)
    return AlphaModel(scatterer, alpha=alpha)
"""
Ultimately we need to have acceess to concentration parameter and mean theta and phi
angles prior might not be the best way to do this given the seperation problem, I'll ask Vinny
instead could use a prior that has these parameters in it but have seperate theta and phi
get around the seperability problem and address it at the model stage (will have to modify
the model anyway). Problem with this is that we need to split off the zero case potentially,
ie) when there is no good mean theta and phi value but maybe k = 0 case can be defined to handle 
this as a specific special case (any angular input should have the same uniform pdf (unbounded though))
"""

"\nUltimately we need to have acceess to concentration parameter and mean theta and phi\nangles prior might not be the best way to do this given the seperation problem, I'll ask Vinny\ninstead could use a prior that has these parameters in it but have seperate theta and phi\nget around the seperability problem and address it at the model stage (will have to modify\nthe model anyway). Problem with this is that we need to split off the zero case potentially,\nie) when there is no good mean theta and phi value but maybe k = 0 case can be defined to handle \nthis as a specific special case (any angular input should have the same uniform pdf (unbounded though))\n"

In [30]:
# try to fit hologram using new prior object

dimer_holo = holo1
#x, y = img.center * SPACING
x, y = Xg_TRUE, Yg_TRUE

# Step 1: Fitting the dimer with gap set to 0
# Set priors and run CMAES allowing z, Theta, Phi and alpha to vary

# I may need to modify the bounds on these priors
z = prior.Uniform(2, 10, name="Z", guess=DIMER_Z_GUESS)
angles = prior.Angles(0.01,np.pi, 2*np.pi, name="Angles")
#theta = prior.Uniform(0-0.1, np.pi+0.1, name="Theta")
#phi = prior.Uniform(0-0.1, 2*np.pi + 0.1, name='Phi')
alpha = prior.Uniform(0.5, 1.2, name="Alpha", guess = 0.8)
step_2_parameters = {'r_1': R_1_MEAN, 'r_2': R_2_MEAN, 'n_1': N_1_MEAN, 'n_2': N_2_MEAN,
                    'x_g': x, 'y_g': y, 'z_g': z,
                    'angles': angles, 'gap': 0, 'alpha': alpha}
model =  create_model_angles(step_2_parameters)
cma_fit_strategy = CmaStrategy(popsize=100, walker_initial_pos=model.generate_guess(scaling=0.5, n=100))
results2 = hp.fit(dimer_holo, model, strategy=cma_fit_strategy)
hp.save(SAVEPATH+'_nogap.h5', results2)

print('von Mises-Fisher angles fit completed')
print(results2.guess_parameters)
print(results2.parameters)

TypeError: cannot unpack non-iterable Angles object

{'Phi': 3.8084479858970015, 'Theta': 3.1330819884262735, 'Z': 4.33783964841273, 'Alpha': 0.8373962779856561}
{'Phi': 3.141592653589793, 'Theta': 1.5707963267948966, 'Z': 4.2, 'Alpha': 0.8}


In [10]:
print(angles)

Angles(concentration_parameter=0.01, theta=3.141592653589793, phi=6.283185307179586, guess=(3.141592653589793, 6.283185307179586), name='Angles')


In [32]:
dimer_holo = holo1
#x, y = img.center * SPACING
x, y = Xg_TRUE, Yg_TRUE

# Step 1: Fitting the dimer with gap set to 0
# Set priors and run CMAES allowing z, Theta, Phi and alpha to vary

# I may need to modify the bounds on these priors
z = prior.Uniform(2, 10, name="Z", guess=DIMER_Z_GUESS)
theta = prior.Uniform(0-0.1, np.pi+0.1, name="Theta")
phi = prior.Uniform(0-0.1, 2*np.pi + 0.1, name='Phi')
alpha = prior.Uniform(0.5, 1.2, name="Alpha", guess = 0.8)
step_2_parameters = {'r_1': R_1_MEAN, 'r_2': R_2_MEAN, 'n_1': N_1_MEAN, 'n_2': N_2_MEAN,
                    'x_g': x, 'y_g': y, 'z_g': z,
                    'theta': theta, 'phi': phi, 'gap': 0, 'alpha': alpha}
model =  create_model(step_2_parameters)

In [33]:
s1_r = step_2_parameters['r_1']
s2_r = step_2_parameters['r_2']
s1_n = step_2_parameters['n_1']
s2_n = step_2_parameters['n_2']
center_x = step_2_parameters['x_g']
center_y = step_2_parameters['y_g']
center_z = step_2_parameters['z_g']
theta = step_2_parameters['theta']
phi = step_2_parameters['phi']
gap = step_2_parameters['gap']
alpha = step_2_parameters['alpha']
    
gap_center = np.array([center_x, center_y, center_z])
components = np.array([np.cos(phi) * np.sin(theta), np.sin(phi) * np.sin(theta), np.cos(theta)])
s1_center = gap_center + (s1_r + gap/2) * components
s2_center = gap_center - (s2_r + gap/2) * components
        
scatterer = Spheres([Sphere(r=s1_r, n=s1_n, center=s1_center),
                    Sphere(r=s2_r, n=s2_n, center=s2_center)], warn=False)

In [34]:
print(gap_center)

[5.0 5.0 Uniform(lower_bound=2, upper_bound=10, guess=4.2, name='Z')]


In [35]:
print(s1_center)

[TransformedPrior(transformation=<built-in function add>, base_prior=(TransformedPrior(transformation=<built-in function mul>, base_prior=(TransformedPrior(transformation=<built-in function mul>, base_prior=(TransformedPrior(transformation=<ufunc 'cos'>, base_prior=(Uniform(lower_bound=-0.1, upper_bound=6.383185307179586, guess=3.141592653589793, name='Phi'),)), TransformedPrior(transformation=<ufunc 'sin'>, base_prior=(Uniform(lower_bound=-0.1, upper_bound=3.241592653589793, guess=1.5707963267948966, name='Theta'),)))), 0.6749016953839639)), 5.0))
 TransformedPrior(transformation=<built-in function add>, base_prior=(TransformedPrior(transformation=<built-in function mul>, base_prior=(TransformedPrior(transformation=<built-in function mul>, base_prior=(TransformedPrior(transformation=<ufunc 'sin'>, base_prior=(Uniform(lower_bound=-0.1, upper_bound=6.383185307179586, guess=3.141592653589793, name='Phi'),)), TransformedPrior(transformation=<ufunc 'sin'>, base_prior=(Uniform(lower_bound=-

In [10]:
print(scatterer.parameters)

{'0:n': 1.5848484802283918, '0:r': 0.6749016953839639, '0:center': [TransformedPrior(transformation=<built-in function add>, base_prior=(TransformedPrior(transformation=<built-in function mul>, base_prior=(TransformedPrior(transformation=<built-in function mul>, base_prior=(TransformedPrior(transformation=<ufunc 'cos'>, base_prior=(Uniform(lower_bound=-0.1, upper_bound=6.383185307179586, guess=3.141592653589793, name='Phi'),)), TransformedPrior(transformation=<ufunc 'sin'>, base_prior=(Uniform(lower_bound=-0.1, upper_bound=3.241592653589793, guess=1.5707963267948966, name='Theta'),)))), 0.6749016953839639)), 5.0)), TransformedPrior(transformation=<built-in function add>, base_prior=(TransformedPrior(transformation=<built-in function mul>, base_prior=(TransformedPrior(transformation=<built-in function mul>, base_prior=(TransformedPrior(transformation=<ufunc 'sin'>, base_prior=(Uniform(lower_bound=-0.1, upper_bound=6.383185307179586, guess=3.141592653589793, name='Phi'),)), TransformedPr

In [36]:
for parameter in scatterer._parameters:
    print(parameter)
    #print(parameter.base_prior)

0:n
0:r
0:center
1:n
1:r
1:center


In [22]:
print(map_keys(model))

NameError: name 'map_keys' is not defined

In [23]:
# lnprior acesses the .lnprob method for each base prior that makes up the 
# transformed priors used in the scatterer object
for parameter in model._parameters:
    print(parameter)

Uniform(lower_bound=-0.1, upper_bound=6.383185307179586, guess=3.141592653589793, name='Phi')
Uniform(lower_bound=-0.1, upper_bound=3.241592653589793, guess=1.5707963267948966, name='Theta')
Uniform(lower_bound=2, upper_bound=10, guess=4.2, name='Z')
Uniform(lower_bound=0.5, upper_bound=1.2, guess=0.8, name='Alpha')


In [ ]:
def create_kaimodel(parameters):
    s1_r = parameters['r_1']
    s2_r = parameters['r_2']
    s1_n = parameters['n_1']
    s2_n = parameters['n_2']
    center_x = parameters['x_g']
    center_y = parameters['y_g']
    center_z = parameters['z_g']
    theta = parameters['theta']
    phi = parameters['phi']
    gap = parameters['gap']
    alpha = parameters['alpha']
    
    gap_center = np.array([center_x, center_y, center_z])
    components = np.array([np.cos(phi) * np.sin(theta), np.sin(phi) * np.sin(theta), np.cos(theta)])
    s1_center = gap_center + (s1_r + gap/2) * components
    s2_center = gap_center - (s2_r + gap/2) * components
        
    scatterer = Spheres([Sphere(r=s1_r, n=s1_n, center=s1_center),
                         Sphere(r=s2_r, n=s2_n, center=s2_center)], warn=False)
    return KaiModel(scatterer, alpha=alpha)